# Exercice 2 — Segmenter, visualiser et surveiller un portefeuille vie (+ mini apprentissage par renforcement)

**Le contexte.** Le directeur du portefeuille veut une *photographie lisible* de ses ~3 500 contrats : quels profils, comment les montrer en une page au comité, et quels contrats « bizarres » faire revoir par un gestionnaire. **Il n'y a pas de cible** : on explore.

| # | Étape | Notion de la session |
|---|---|---|
| Q1 | Pourquoi standardiser | distance, dépendance à l'échelle |
| Q2 | Choisir k | silhouette, stabilité, sens métier |
| Q3 | Nommer les segments | grille d'interprétation à 4 colonnes |
| Q4 | PCA | variance expliquée, loadings, perte d'information |
| Q5 | Un cluster porté par une seule variable | piège du non supervisé |
| Q6 | Anomalies | distance au centre, Isolation Forest, revue experte |
| Q7 | Mini Q-learning | agent, récompense, γ, politique |
| Q8 | Quiz de consolidation | quand utiliser / éviter |

> **Mode d'emploi.** Remplacez chaque `...` par votre code, exécutez la cellule `verifier(...)` suivante. Les 💡 indices sont repliables.

## Préparation (cellule fournie — à exécuter)

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

import hashlib
RES = {}
def verifier(qid, test, ok="Bravo !", ko="Pas encore — relis l'énoncé ou ouvre l'indice."):
    """Auto-correction : exécute `test()` et enregistre le résultat de la question `qid`."""
    try:
        res = bool(test()); detail = ""
    except Exception as e:
        res = False; detail = f" [{type(e).__name__} : {e}]"
    RES[qid] = res
    print(("✅ " + ok) if res else ("❌ " + ko + detail))

def qcm(qid, reponse, empreinte):
    ok = hashlib.sha256(f"{qid}:{str(reponse).strip().upper()}".encode()).hexdigest()[:10] == empreinte
    RES[qid] = ok
    print("✅ Bonne réponse !" if ok else "❌ Ce n'est pas la bonne réponse — relis la slide correspondante.")

def bilan(attendus):
    print("BILAN DE L'EXERCICE")
    for q in attendus: print(f"  {q:<6}", "✅" if RES.get(q) else "⬜ à faire / à revoir")
    print(f"\nScore : {sum(RES.get(q, False) for q in attendus)}/{len(attendus)}")
import numpy as np, pandas as pd

def generer_portefeuille_seg(seed=7):
    """Portefeuille vie synthétique (4 profils latents + quelques contrats atypiques injectés)."""
    rng = np.random.default_rng(seed)
    def bloc(n, age, anc, enc, vers, arb, rach):
        a = np.clip(rng.normal(*age, n), 20, 90).round()
        e = np.exp(rng.normal(*enc, n)).round(-2)
        return pd.DataFrame(dict(age=a, anciennete=np.clip(rng.normal(*anc, n), 0.2, 30).round(1), encours=e,
            versements_12m=np.where(rng.random(n) < vers[0], e*rng.uniform(.03, vers[1], n), 0).round(-1),
            nb_arbitrages_12m=rng.poisson(arb, n), nb_rachats_passes=rng.poisson(rach, n)))
    df = pd.concat([bloc(1100, (34,5), (2.5,1.3), (9.6,.5), (.9,.25), 1.5, .3),
                    bloc(800,  (62,7), (10,4),   (12.0,.5), (.2,.08), .2, .05),
                    bloc(700,  (52,9), (8.3,.6), (10.8,.6), (.05,.06), .5, .8),
                    bloc(900,  (71,6), (15,5),   (10.4,.6), (.02,.06), .05, .1)], ignore_index=True)
    ano = pd.DataFrame(dict(
        age=[30]*12 + [58]*10 + [49]*8, anciennete=[27 + i*.2 for i in range(12)] + [9]*10 + [6]*8,
        encours=[0]*12 + [1_200_000 + 60_000*i for i in range(10)] + [30_000]*8,
        versements_12m=[0]*12 + [0]*10 + [250_000 + 20_000*i for i in range(8)],
        nb_arbitrages_12m=[0]*12 + [8]*10 + [1]*8, nb_rachats_passes=[0]*12 + [7]*10 + [0]*8))
    df = pd.concat([df, ano], ignore_index=True)
    ids_ano = set(range(len(df) - len(ano), len(df)))
    idx = rng.permutation(len(df))                       # mélange : la position ne trahit pas les anomalies
    df = df.iloc[idx].reset_index(drop=True)
    df.insert(0, "contrat_id", np.arange(1, len(df) + 1))
    ids = {int(i) + 1 for i, old in enumerate(idx) if old in ids_ano}
    return df, ids

df, _IDS_ANOMALIES = generer_portefeuille_seg()      # _IDS_ANOMALIES : vérité cachée, utilisée seulement par les vérifications
VARS = ["age", "anciennete", "encours", "versements_12m", "nb_arbitrages_12m", "nb_rachats_passes"]
print(f"{len(df):,} contrats, {len(VARS)} variables, aucune cible.")
df.head()

3,530 contrats, 6 variables, aucune cible.


   contrat_id   age  anciennete   encours  versements_12m  nb_arbitrages_12m  \
0           1 30.00        5.30 21,600.00        2,180.00                  2   
1           2 39.00        4.80 21,400.00        1,370.00                  1   
2           3 40.00        8.40 49,800.00            0.00                  1   
3           4 68.00        9.10 35,300.00            0.00                  0   
4           5 76.00       11.10 17,500.00            0.00                  0   

   nb_rachats_passes  
0                  0  
1                  0  
2                  1  
3                  0  
4                  0  

### Dictionnaire de données
| Variable | Description |
|---|---|
| `age` | Âge de l'assuré (années) |
| `anciennete` | Ancienneté du contrat (années) |
| `encours` | Provision mathématique (€) — distribution très asymétrique |
| `versements_12m` | Versements des 12 derniers mois (€) |
| `nb_arbitrages_12m` | Nombre d'arbitrages sur 12 mois |
| `nb_rachats_passes` | Nombre de rachats passés |

Pour que les euros ne dominent pas, `encours` et `versements_12m` sont passés en `log1p` (fourni ci-dessous) : les variables en € suivent une loi très asymétrique.

## Q1 · Pourquoi standardiser avant un k-means ?
**Notion :** *sans standardisation, l'encours écrase l'âge dans la distance : les clusters suivent l'euro, pas le comportement.*

1. Créez `X_std` : `StandardScaler().fit_transform(...)` appliqué à `X_log` (variables `VARS`, avec `encours` et `versements_12m` en log).
2. Entraînez deux k-means (`n_clusters=4, n_init=10, random_state=0`) : `km_brut` sur `df[VARS]` en euros bruts, et `km` sur `X_std`.
3. La fonction `part_variance_encours(labels)` (fournie) donne la part de la variance de l'`encours` expliquée par les clusters. Comparez `eta_brut` et `eta_std`.


<details><summary>💡 <b>Indice</b></summary>

`km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X)` ; les étiquettes sont dans `km.labels_`.

</details>

In [2]:
X_log = df[VARS].copy()
X_log["encours"] = np.log1p(X_log["encours"]); X_log["versements_12m"] = np.log1p(X_log["versements_12m"])

def part_variance_encours(labels):
    e = df["encours"]; labels = np.asarray(labels)
    intra = sum(((e[labels == c] - e[labels == c].mean()) ** 2).sum() for c in np.unique(labels))
    return 1 - intra / ((e - e.mean()) ** 2).sum()

X_std = StandardScaler().fit_transform(X_log)
km_brut = KMeans(n_clusters=4, n_init=10, random_state=0).fit(df[VARS])
km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X_std)
eta_brut = part_variance_encours(km_brut.labels_)
eta_std = part_variance_encours(km.labels_)
print(f"Part de variance de l'encours expliquée : bruts {eta_brut:.0%} | standardisés {eta_std:.0%}")

Part de variance de l'encours expliquée : bruts 91% | standardisés 23%


In [3]:
verifier("Q1", lambda: abs(X_std.mean()) < 1e-6 and abs(X_std.std() - 1) < 1e-6 and eta_brut > 0.8 and eta_std < 0.5,
         "Sur données brutes, les clusters ne sont qu'un découpage de l'encours (>80 % de sa variance) ; standardisés, ils reflètent plusieurs dimensions.",
         "X_std doit être centrée-réduite, et km_brut/km doivent être des KMeans(4) entraînés sur les bonnes données.")

✅ Sur données brutes, les clusters ne sont qu'un découpage de l'encours (>80 % de sa variance) ; standardisés, ils reflètent plusieurs dimensions.


**Correction commentée :** sur les euros bruts, k-means ne voit que l'encours (les 6 variables ont des échelles de 1 à 10⁶). La standardisation rend les variables comparables ; le `log1p` limite en plus l'influence de la queue de distribution.

## Q2 · Choisir k : silhouette, stabilité, sens métier
**Notion :** *la silhouette propose, le métier dispose.*

1. Pour `k` de 2 à 8, calculez la **silhouette moyenne** de `KMeans(k, n_init=10, random_state=0)` sur `X_std` → dictionnaire `sil`. `k_geo` = k qui la maximise.
2. **Stabilité** (k = 4) : pour 5 graines `b` (0 à 4), tirez 80 % des contrats sans remise (`rng = np.random.default_rng(b)`, `rng.choice(len(X_std), int(0.8*len(X_std)), replace=False)`), entraînez un k-means sur cet échantillon, prédisez les étiquettes de **tous** les contrats et calculez l'`adjusted_rand_score` contre `km.labels_`. `ari_moyen` = moyenne.
3. Quel k retenez-vous ? `K_METIER` (le choix documenté : ici, on veut isoler le profil « pré-échéance fiscale », donc **4**).


<details><summary>💡 <b>Indice</b></summary>

`silhouette_score(X_std, labels)` ; ARI proche de 1 = segments qui survivent au rééchantillonnage.

</details>

In [4]:
sil = {}
for k in range(2, 9):
    lab = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X_std)
    sil[k] = silhouette_score(X_std, lab)
k_geo = max(sil, key=sil.get)

aris = []
for b in range(5):
    rng = np.random.default_rng(b)
    idx = rng.choice(len(X_std), int(0.8 * len(X_std)), replace=False)
    km_b = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X_std[idx])
    aris.append(adjusted_rand_score(km.labels_, km_b.predict(X_std)))
ari_moyen = np.mean(aris)
K_METIER = 4
print({k: round(v, 3) for k, v in sil.items()}, "| k géométrique :", k_geo, "| ARI moyen :", round(ari_moyen, 3))

{2: 0.433, 3: 0.377, 4: 0.323, 5: 0.33, 6: 0.332, 7: 0.296, 8: 0.267} | k géométrique : 2 | ARI moyen : 0.925


In [5]:
verifier("Q2a", lambda: set(sil) == set(range(2, 9)) and k_geo == max(sil, key=sil.get) and 0 < sil[2] < 1, "Balayage de k correct.")
verifier("Q2b", lambda: ari_moyen > 0.8, "Segments stables au rééchantillonnage (ARI proche de 1).")
verifier("Q2c", lambda: K_METIER == 4, "k = 4 : choix métier assumé, même si k_geo (silhouette) est plus petit — à documenter comme un choix, pas un résultat automatique.",
         "Relisez l'énoncé : on veut isoler le segment « pré-échéance fiscale ».")
plt.plot(list(sil), list(sil.values()), "o-"); plt.axvline(K_METIER, ls=":", c="r"); plt.xlabel("k"); plt.ylabel("silhouette"); plt.title("Silhouette moyenne selon k"); plt.show()

✅ Balayage de k correct.
✅ Segments stables au rééchantillonnage (ARI proche de 1).
✅ k = 4 : choix métier assumé, même si k_geo (silhouette) est plus petit — à documenter comme un choix, pas un résultat automatique.


<Figure size 640x480 with 1 Axes>

## Q3 · Profiler et nommer les segments
**Notion :** *un cluster qu'on ne peut pas nommer en langage métier est inutilisable.*

1. `profil` : moyenne des 6 variables (en unités d'origine) par cluster de `km`, avec la colonne `effectif`.
2. `profil_z` : écart de chaque moyenne de cluster à la moyenne du portefeuille, en nombre d'écarts-types → repérez les variables dominantes.
3. Donnez un **nom** à chaque cluster dans `NOMS` (dictionnaire `{numéro: "nom"}`) et complétez la grille à 4 colonnes en Markdown plus bas.


<details><summary>💡 <b>Indice</b></summary>

- `df.assign(cluster=km.labels_).groupby("cluster")[VARS].mean()`
- `profil_z = (profil[VARS] - df[VARS].mean()) / df[VARS].std()`

</details>

In [6]:
dfc = df.assign(cluster=km.labels_)
profil = dfc.groupby("cluster")[VARS].mean()
profil["effectif"] = dfc.groupby("cluster").size()
profil_z = (profil[VARS] - df[VARS].mean()) / df[VARS].std()
NOMS = {}
for c in profil.index:
    r = profil.loc[c]
    if 7 <= r["anciennete"] <= 9.5 and r["nb_rachats_passes"] > 1: NOMS[c] = "pré-échéance fiscale"
    elif r["age"] < 42: NOMS[c] = "jeunes dynamiques"
    elif r["encours"] > profil["encours"].median() and r["age"] < 66: NOMS[c] = "patrimoniaux stables"
    else: NOMS[c] = "seniors dormants"

In [7]:
display(profil.round(1)); display(profil_z.round(2))
verifier("Q3", lambda: set(NOMS) == set(profil.index) and len(set(NOMS.values())) == 4 and all(len(str(v)) > 3 and v != "..." for v in NOMS.values())
         and profil_z.shape == (4, 6),
         "4 segments, 4 noms distincts. Vérifiez que chacun se raconte en une phrase.")

          age  anciennete    encours  versements_12m  nb_arbitrages_12m  \
cluster                                                                   
0       33.90        2.60  17,228.70        4,518.70               1.60   
1       60.60        9.20 140,759.00        1,145.60               0.30   
2       51.10        8.00  87,973.50          177.90               0.70   
3       70.40       16.60  36,730.30           29.90               0.00   

         nb_rachats_passes  effectif  
cluster                               
0                     0.30      1092  
1                     0.00      1221  
2                     1.60       441  
3                     0.10       776  

          age  anciennete  encours  versements_12m  nb_arbitrages_12m  \
cluster                                                                 
0       -1.20       -1.02    -0.50            0.17               0.81   
1        0.45        0.09     0.60           -0.04              -0.37   
2       -0.14       -0.10     0.13           -0.11               0.06   
3        1.05        1.35    -0.32           -0.12              -0.58   

         nb_rachats_passes  
cluster                     
0                    -0.07  
1                    -0.41  
2                     1.82  
3                    -0.29  

✅ 4 segments, 4 noms distincts. Vérifiez que chacun se raconte en une phrase.


**Grille d'interprétation (exemple de correction) :**

| Segment | Variables dominantes | Usage potentiel | Risque de mauvaise lecture |
|---|---|---|---|
| jeunes dynamiques | âge < 40, versements réguliers, arbitrages | ciblage des offres, conseil | effet de cohorte confondu avec l'âge |
| patrimoniaux stables | gros encours, faible rotation | gestion ALM, fidélisation | hétérogène en garanties |
| pré-échéance fiscale | ancienneté 7–9 ans, rachats passés élevés | anticipation des rachats | segment quasi entièrement défini par une variable (l'ancienneté) |
| seniors dormants | âge élevé, ancienneté longue, aucun mouvement | suivi des décès / déshérence | ne pas en déduire un faible risque de rachat : ce n'est pas la cible |

## Q4 · PCA : projeter pour voir, en assumant la perte
**Notion :** *la PCA conserve de la variance, pas du sens.*

1. `pca = PCA().fit(X_std)`. Calculez `var_cumulee` (variance expliquée cumulée) et `n_80` : nombre minimal de composantes pour expliquer au moins **80 %** de la variance.
2. `loadings` : DataFrame des coefficients (`pca.components_.T`) des 2 premières composantes, indexé par `VARS`, colonnes `["PC1", "PC2"]`.
3. Lisez PC1 : quelles variables s'opposent ? Sous quelle étiquette métier la décririez-vous ?


<details><summary>💡 <b>Indice</b></summary>

`np.cumsum(pca.explained_variance_ratio_)` ; `n_80 = int(np.argmax(var_cumulee >= 0.80)) + 1`.

</details>

In [8]:
pca = PCA().fit(X_std)
var_cumulee = np.cumsum(pca.explained_variance_ratio_)
n_80 = int(np.argmax(var_cumulee >= 0.80)) + 1
loadings = pd.DataFrame(pca.components_[:2].T, index=VARS, columns=["PC1", "PC2"])
Z = pca.transform(X_std)[:, :2]

In [9]:
verifier("Q4", lambda: abs(var_cumulee[-1] - 1) < 1e-9 and n_80 == int(np.argmax(np.cumsum(PCA().fit(X_std).explained_variance_ratio_) >= 0.8)) + 1
         and loadings.shape == (6, 2), f"{n_80} composantes suffisent pour 80 % de variance : la projection 2D en conserve {var_cumulee[1]:.0%} (le reste est perdu).")
display(loadings.round(2))
plt.scatter(Z[:, 0], Z[:, 1], c=km.labels_, s=6, cmap="tab10"); plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("Portefeuille en 2D (couleur = cluster)"); plt.show()

✅ 4 composantes suffisent pour 80 % de variance : la projection 2D en conserve 66% (le reste est perdu).


                    PC1   PC2
age                0.53  0.04
anciennete         0.48 -0.04
encours            0.30  0.37
versements_12m    -0.47 -0.22
nb_arbitrages_12m -0.41  0.31
nb_rachats_passes -0.10  0.85

<Figure size 640x480 with 1 Axes>

**Correction commentée :** PC1 oppose les contrats **anciens / âgés** (âge, ancienneté) aux contrats **actifs** (versements, arbitrages) : c'est un axe « maturité du contrat vs activité ». PC2 est dominé par le comportement de rachat passé et l'encours. Une projection 2D ne conserve qu'une partie de la variance : ne tirez aucune conclusion causale de la forme du nuage.

## Q5 · Piège : un cluster porté par une seule variable
**Notion (exercice facultatif de la session) :** *retirez `anciennete` et observez le segment « pré-échéance fiscale » disparaître.*

1. Refaites tout le pipeline **sans** `anciennete` : `X_log_sans`, `X_std_sans` (nouveau `StandardScaler`), `km_sans` (k = 4, mêmes réglages).
2. Calculez `ari_sans` = ARI entre `km.labels_` et `km_sans.labels_`.
3. Interprétez : l'ARI vaut 1 si les deux segmentations sont identiques ; quelle part de la structure repose sur `anciennete` ?


<details><summary>💡 <b>Indice</b></summary>

`adjusted_rand_score(km.labels_, km_sans.labels_)` ; un ARI < 0,8 signale une segmentation qui change nettement.

</details>

In [10]:
X_log_sans = X_log.drop(columns="anciennete")
X_std_sans = StandardScaler().fit_transform(X_log_sans)
km_sans = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X_std_sans)
ari_sans = adjusted_rand_score(km.labels_, km_sans.labels_)
print(f"ARI avec/sans ancienneté : {ari_sans:.2f}")

ARI avec/sans ancienneté : 0.64


In [11]:
verifier("Q5", lambda: X_log_sans.shape[1] == 5 and abs(ari_sans - adjusted_rand_score(km.labels_, km_sans.labels_)) < 1e-9 and ari_sans < 0.8,
         "La segmentation change nettement sans l'ancienneté : une partie de la structure reposait sur une seule variable.")

✅ La segmentation change nettement sans l'ancienneté : une partie de la structure reposait sur une seule variable.


**Correction commentée :** l'ARI chute (≈ 0,6) : environ un tiers de la segmentation se réorganise, car l'ancienneté portait à elle seule la frontière du segment « pré-échéance ». Leçon : tester la stabilité **aussi à la composition des variables**, et ne pas présenter un cluster comme une vérité.

## Q6 · Anomalies : prioriser l'attention humaine
**Notion :** *score = distance au plus proche centre ; Isolation Forest ; le score classe par étrangeté, pas par importance.*

1. `dist` : pour chaque contrat, distance au centre le plus proche (`km.transform(X_std).min(axis=1)`). `top_km` = positions des **30** contrats les plus éloignés.
2. `iso = IsolationForest(n_estimators=200, random_state=0).fit(X_std)` ; `score_iso = -iso.score_samples(X_std)` (plus grand = plus atypique). `top_iso` = positions des 30 plus atypiques.
3. `recoupement` : nombre de contrats communs aux deux listes.


<details><summary>💡 <b>Indice</b></summary>

`np.argsort(-score)[:30]` donne les 30 positions aux plus grands scores.

</details>

In [12]:
dist = km.transform(X_std).min(axis=1)
top_km = np.argsort(-dist)[:30]
iso = IsolationForest(n_estimators=200, random_state=0).fit(X_std)
score_iso = -iso.score_samples(X_std)
top_iso = np.argsort(-score_iso)[:30]
recoupement = len(set(top_km) & set(top_iso))
print("Contrats communs aux deux méthodes :", recoupement, "/ 30")

Contrats communs aux deux méthodes : 24 / 30


In [13]:
def _vrais(top): return len({int(df["contrat_id"].iloc[i]) for i in top} & _IDS_ANOMALIES)
verifier("Q6a", lambda: len(top_km) == 30 and _vrais(top_km) >= 15, f"Score de distance : {_vrais(top_km)}/30 des contrats signalés sont des anomalies injectées.")
verifier("Q6b", lambda: len(top_iso) == 30 and _vrais(top_iso) >= 15, f"Isolation Forest : {_vrais(top_iso)}/30 des contrats signalés sont des anomalies injectées.")
verifier("Q6c", lambda: recoupement == len(set(top_km) & set(top_iso)) and recoupement >= 10, "Les deux méthodes convergent largement : le consensus est un bon candidat à la revue experte.")
df.iloc[top_km[:8]][["contrat_id"] + VARS]

✅ Score de distance : 22/30 des contrats signalés sont des anomalies injectées.
✅ Isolation Forest : 22/30 des contrats signalés sont des anomalies injectées.
✅ Les deux méthodes convergent largement : le consensus est un bon candidat à la revue experte.


      contrat_id   age  anciennete      encours  versements_12m  \
1322        1323 58.00        9.00 1,740,000.00            0.00   
1626        1627 58.00        9.00 1,680,000.00            0.00   
1107        1108 58.00        9.00 1,620,000.00            0.00   
2715        2716 58.00        9.00 1,560,000.00            0.00   
2318        2319 58.00        9.00 1,500,000.00            0.00   
483          484 58.00        9.00 1,440,000.00            0.00   
2149        2150 58.00        9.00 1,380,000.00            0.00   
1016        1017 58.00        9.00 1,320,000.00            0.00   

      nb_arbitrages_12m  nb_rachats_passes  
1322                  8                  7  
1626                  8                  7  
1107                  8                  7  
2715                  8                  7  
2318                  8                  7  
483                   8                  7  
2149                  8                  7  
1016                  8             

**Correction commentée :** on distingue deux familles : **incohérences** (ancienneté > âge − 18, encours nul : erreur d'alimentation à corriger à la source) et **comportements atypiques réels** (rachats répétés sur ~1 M€, versements disproportionnés à l'encours : à étudier, voire signal de fraude/blanchiment). Un score élevé n'est ni une erreur ni un risque en soi : c'est un outil de **priorisation**, la décision reste métier.

## Q7 · Mini apprentissage par renforcement : le monde-grille actuariel
**Notions :** agent · environnement · récompense · politique · retour actualisé (γ) · Q-learning · exploration / exploitation.

Chaque case de la grille 4 × 4 est un **état de marché**, chaque mouvement (↑ ↓ ← →) une **décision d'allocation**. Départ en haut à gauche ; les cases rouges `R` sont des **chocs de taux** (récompense −10, fin d'épisode), la case verte `G` l'objectif de rendement ajusté du risque (+10, fin). Chaque pas coûte −1 (coût de portage / de liquidité).

**7a — Retour actualisé.** Écrivez `retour(recompenses, gamma)` = Σ γᵗ·rₜ (ce n'est autre qu'une valeur actuelle avec γ comme facteur d'actualisation).

**7b — Mise à jour de Q.** Dans la boucle, écrivez la règle de Q-learning :
`Q[s, a] ← Q[s, a] + α · ( r + γ · max_a' Q[s', a'] − Q[s, a] )` (sans le terme `max` si l'épisode est terminé).


<details><summary>💡 <b>Indice</b></summary>

Si `fini` : `cible = r` ; sinon `cible = r + gamma * Q[s2].max()`. Puis `Q[s, a] += alpha * (cible - Q[s, a])`.

</details>

In [14]:
def retour(recompenses, gamma):
    return sum(gamma ** t * r for t, r in enumerate(recompenses))

MOUVEMENTS = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # ↑ ↓ ← →
CHOCS, OBJECTIF, N = {(1, 1), (2, 3)}, (3, 3), 4

def pas(etat, a):
    """Environnement (fourni) : renvoie (nouvel état, récompense, fini)."""
    i, j = etat[0] + MOUVEMENTS[a][0], etat[1] + MOUVEMENTS[a][1]
    nouveau = (min(max(i, 0), N - 1), min(max(j, 0), N - 1))
    if nouveau in CHOCS: return nouveau, -10, True
    if nouveau == OBJECTIF: return nouveau, 10, True
    return nouveau, -1, False

def entrainer(episodes=3000, alpha=0.1, gamma=0.95, eps=0.2, graine=0):
    rng = np.random.default_rng(graine)
    Q = np.zeros((N, N, 4))
    for _ in range(episodes):
        s, fini, t = (0, 0), False, 0
        while not fini and t < 100:
            a = rng.integers(4) if rng.random() < eps else int(np.argmax(Q[s]))    # ε-greedy : exploration / exploitation
            s2, r, fini = pas(s, a)
            cible = r if fini else r + gamma * Q[s2].max()
            Q[s][a] += alpha * (cible - Q[s][a])
            s, t = s2, t + 1
    return Q

In [15]:
verifier("Q7a", lambda: abs(retour([1, 1, 1], 0.5) - 1.75) < 1e-9 and abs(retour([5, 5], 1.0) - 10) < 1e-9,
         "Retour actualisé correct : γ = 0,5 pondère très fortement le présent (1 + 0,5 + 0,25 = 1,75).")
Q = entrainer()

def politique_gloutonne(Q):
    s, chemin, recomp = (0, 0), [(0, 0)], []
    for _ in range(20):
        s, r, fini = pas(s, int(np.argmax(Q[s]))); chemin.append(s); recomp.append(r)
        if fini: break
    return chemin, recomp
chemin, recomp = politique_gloutonne(Q)
print("Chemin appris :", chemin, "| retour (γ=0,95) :", round(retour(recomp, 0.95), 2))
verifier("Q7b", lambda: chemin[-1] == OBJECTIF and not (set(chemin) & CHOCS) and len(chemin) <= 9,
         "La politique apprise atteint l'objectif en évitant les chocs, par le chemin le plus court.")

✅ Retour actualisé correct : γ = 0,5 pondère très fortement le présent (1 + 0,5 + 0,25 = 1,75).


Chemin appris : [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (3, 3)] | retour (γ=0,95) : 3.21
✅ La politique apprise atteint l'objectif en évitant les chocs, par le chemin le plus court.


**7c — Correction commentée.** (i) γ faible rend l'agent myope : le pas coûte −1 tout de suite, l'objectif +10 est « loin », donc l'agent hésite à s'y engager ; avec ε = 0, aucune exploration : l'agent peut se figer sur un chemin médiocre découvert au hasard. (ii) Non : pas de simulateur validé, récompense scalaire (un seul indicateur) alors que les objectifs vie sont multi-critères (rendement, SCR, liquidité), et le passé ne « répond » pas aux actions de l'agent. Le RL reste exploratoire ; les méthodes de scénarios classiques restent la réponse opérationnelle.

## Q8 · Quiz de consolidation
Répondez par `A`, `B` ou `C`.

**Q8a.** Pourquoi un cluster n'a-t-il pas automatiquement un sens métier ?
- **A** — Parce que k-means est un algorithme instable.
- **B** — Parce qu'il regroupe par ressemblance géométrique : le sens se construit par interprétation et validation métier.
- **C** — Parce que les clusters sont toujours aléatoires.

**Q8b.** Comment évalue-t-on un modèle non supervisé, faute de cible ?
- **A** — Par l'AUC sur un jeu de test.
- **B** — Par la stabilité (rééchantillonnage), la couverture et l'utilité métier.
- **C** — Par le nombre de clusters : plus il y en a, mieux c'est.

**Q8c.** Un score d'anomalie très élevé signifie :
- **A** — Qu'il faut corriger automatiquement le contrat.
- **B** — Que le contrat est une fraude.
- **C** — Que le contrat est atypique : il mérite une revue experte, pas un verdict.

**Q8d.** Que maximise la PCA, et que perd-elle ?
- **A** — Elle maximise la variance conservée par axe, mais perd la signification directe des variables.
- **B** — Elle maximise la précision de prédiction, mais perd des observations.
- **C** — Elle maximise le nombre de clusters, mais perd la standardisation.

In [16]:
rep_q8a, rep_q8b, rep_q8c, rep_q8d = "B", "B", "C", "A"
qcm("Q8a", rep_q8a, "8b59912273"); qcm("Q8b", rep_q8b, "bb2ba963f4"); qcm("Q8c", rep_q8c, "a4cebcf624"); qcm("Q8d", rep_q8d, "e4305b801f")

✅ Bonne réponse !
✅ Bonne réponse !
✅ Bonne réponse !
✅ Bonne réponse !


## Bilan

In [17]:
bilan(["Q1", "Q2a", "Q2b", "Q2c", "Q3", "Q4", "Q5", "Q6a", "Q6b", "Q6c", "Q7a", "Q7b", "Q8a", "Q8b", "Q8c", "Q8d"])

BILAN DE L'EXERCICE
  Q1     ✅
  Q2a    ✅
  Q2b    ✅
  Q2c    ✅
  Q3     ✅
  Q4     ✅
  Q5     ✅
  Q6a    ✅
  Q6b    ✅
  Q6c    ✅
  Q7a    ✅
  Q7b    ✅
  Q8a    ✅
  Q8b    ✅
  Q8c    ✅
  Q8d    ✅

Score : 16/16
